In [ ]:
#!pip install numpy #V1.23.5
#!pip install pandas #v2.0.3
#!pip install keras #V2.6.0
#!pip install tensorflow #V2.6.0
#!pip install scikit-learn #V1.3.2
#!pip install fastparquet #V2024.2.0
#!pip install seaborn

In [ ]:
import os
import pandas as pd
import keras
import tensorflow
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from keras.layers import Dense, LSTM
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
pd.set_option('display.max_columns', None) #set to none to show all columns
data_path = '' #PATH REMOVED -- SET YOUR OWN DATA PATH HERE
df = pd.read_parquet(data_path) #df = data frame
print(df.head())

In [ ]:
unique_labels = df['Label'].unique()
label_amount = df['Label'].value_counts()
print(label_amount)
print(unique_labels)

In [ ]:
unique_class_labels = df['ClassLabel'].unique()
classlabel_amount = df['ClassLabel'].value_counts()
print(classlabel_amount)
print(unique_class_labels)

In [ ]:
samples_per_class = 10000 #can be changed if higher classes are overfitting
boosted_classes = ['Benign', 'Infiltration']
boosted_cap = 20000
new_df = pd.DataFrame() #new dataframe to add limited classes to, allows option to delete old dataframe if memory is needed.

for label in df['ClassLabel'].unique(): #for loop that iterates and checks over each unique class label.
    class_samples = df[df['ClassLabel'] == label] #checks if that class label is the same as its checking, i.e. checks if its DDoS and if true, adds itto class_samples.
    cap = boosted_cap if label in boosted_classes else samples_per_class
    if len(class_samples) > cap: #if larger than max samples we want..
        class_samples = class_samples.sample(cap, random_state=1) #the sample function randomly selects specified amount of rows, random_state ensures reproducibility, if not added will provide different rows each time. Important in ML to see if changes are making results not different data.
    new_df = pd.concat([new_df, class_samples], ignore_index=True) #concatenates class_samples to our new_df frame. ignore_index allows for clean index, easier to refer to rows later.

print(new_df.head())  #to check if data columns are still preserved/looks the same.
print(new_df['ClassLabel'].value_counts())

In [ ]:
new_df = new_df.drop(['Label'], axis=1)
cols_to_drop = ['Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean',
    'Active Std', 'Active Max', 'Active Min', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Total Fwd Packets',
    'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Bwd IAT Total','Fwd IAT Total', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
new_df = new_df.drop(columns=cols_to_drop)
df_encoded = pd.get_dummies(new_df, columns=['ClassLabel']) 

In [ ]:
numerical_features = ['Flow Duration', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 
    'Fwd Packet Length Std', 'Bwd Packet Length Max','Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean',
    'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Mean', 
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Packets/s', 'Bwd Packets/s','Packet Length Max', 
    'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance','SYN Flag Count', 'URG Flag Count', 'Avg Packet Size','Avg Fwd Segment Size', 'Avg Bwd Segment Size']

scaler = MinMaxScaler()
df_encoded[numerical_features] = scaler.fit_transform(df_encoded[numerical_features])
print(df_encoded.head())

In [ ]:
X = df_encoded.loc[:, ~df_encoded.columns.str.match('^ClassLabel_')] 
y = df_encoded.loc[:, df_encoded.columns.str.match('^ClassLabel_')]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=1) #splits data in 0.2 so 80/20, 80% goes to x and y train, 20% goes to temp.
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=1)# 20% temp gets split 0.5 so 50/50, 10% each to val and test.

In [ ]:
X_train = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_val = X_val.values.reshape((X_val.shape[0], 1, X_val.shape[1]))
X_test = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
#change integer labels from one hot encoding for class weights
y_train_labels = np.argmax(y_train, axis=1)

#class weights to penalise the model more when it misclassifies underrepresented classes like benign and infiltration
#class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)
#class_weight_dict = dict(enumerate(class_weights))
#in order as benign, botnet, bruteforce, ddos, dos, infiltration, porstcan, webattack.
class_weight_dict = {0: 2.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 2.0, 6: 3.0, 7: 2.5,}

improved_model = Sequential()
improved_model.add(Bidirectional(LSTM(512, activation='tanh', return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])))
improved_model.add(Dropout(0.1)) #randomly disables 10% of neurons during training so the model cant rely on one pathway
improved_model.add(Bidirectional(LSTM(256, activation='tanh', return_sequences=True)))
improved_model.add(Dropout(0.1))
improved_model.add(Bidirectional(LSTM(256, activation='tanh')))
improved_model.add(Dropout(0.1))
improved_model.add(Dense(128, activation='relu')) #layer to combine outputs so far before classifying. 
improved_model.add(Dropout(0.1))
improved_model.add(Dense(y_train.shape[1], activation='softmax'))
improved_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

#callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
#halves learning rate if val_loss doesnt improve for 3 epochs
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

improved_history = improved_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=200, batch_size=64, class_weight=class_weight_dict, callbacks=[early_stopping, reduce_lr])

In [ ]:
val_loss, val_acc   = improved_model.evaluate(X_val,  y_val,  verbose=0)
test_loss, test_acc = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"Validation Loss: {val_loss:.4f} Accuracy: {val_acc:.4f}")
print(f"Test Loss: {test_loss:.4f} Accuracy: {test_acc:.4f}")

y_test_prob = improved_model.predict(X_test) #runs test data and finds probabilities
y_test_pred = np.argmax(y_test_prob, axis=1) #the prediction data
y_test_true = np.argmax(y_test.values, axis=1) #the true values to compare it to
class_names = [col.replace("ClassLabel_", "") for col in y.columns]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Improved Model - Training History", fontsize=15)

axes[0].plot(improved_history.history["loss"], label="Train Loss")
axes[0].plot(improved_history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(improved_history.history["accuracy"], label="Train Accuracy")
axes[1].plot(improved_history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
plt.show()

In [ ]:
print("Classification Report (Test Data)")
print(classification_report(y_test_true, y_test_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_test_true, y_test_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Improved Model - Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()